In [1]:
import torch 
import torch.nn as nn
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import torchvision 
import torchvision.transforms as transforms
import time    
import os
from  pathlib import Path
from torch.utils.data import DataLoader ,Dataset

from PIL import Image


In [2]:
print("PyTorch Version: ",torch.__version__)
print("GPU Available: ",torch.cuda.is_available())
print("GPU Count: ",torch.cuda.device_count())
if torch.cuda.is_available():
    print("The type of the GPU: ",torch.cuda.get_device_name(0))

PyTorch Version:  2.12.1+cpu
GPU Available:  False
GPU Count:  0


In [1]:
import kagglehub
from pathlib import Path

dataset_path = Path(''
    # kagglehub.dataset_download("moltean/fruits")
)
ROOT = Path(r"C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99")
print(ROOT.exists())
# print(dataset_path)

True


C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
for item in ROOT.iterdir():
    print(item)
    for i in item.iterdir():
        print(i)

C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_100x100
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_100x100\fruits-360
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_3-body-problem
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_3-body-problem\fruits-360-3-body-problem
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_meta
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_meta\fruits-360-meta
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_multi
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_multi\LICENSE
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_multi\README.md
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_multi\test-multiple_fruits
C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fru

In [4]:
from pathlib import Path

DATASET_PATH = Path(r"C:\Users\PC\.cache\kagglehub\datasets\moltean\fruits\versions\99\fruits-360_100x100/fruits-360")

TRAIN_PATH = DATASET_PATH / "Training"
TEST_PATH = DATASET_PATH / "Test"
print("Dataset Path: ", DATASET_PATH.exists())
print("Training Path: ", TRAIN_PATH.exists()
      )
print("Test Path: ", TEST_PATH.exists()
      )

Dataset Path:  True
Training Path:  True
Test Path:  True


In [9]:
import warnings


class FruitsDataset(Dataset):
    def __init__(self, root_dir, transform=None,mode='train',image_extensions =None):
        """Initialize the fruits dataset."""
        super().__init__()
        self.mode = mode
        self.image_extensions = image_extensions
        if image_extensions is None:
            self.image_extensions =  [".jpg",".png",".jpeg",".bmp",".webp"]

        self.root_dir = Path(root_dir)
        
        if not self.root_dir.is_dir():
            raise ValueError(f"Root directory {root_dir} does not exist.")
        
        self.class_names = sorted([f.name for f in self.root_dir.iterdir() if f.is_dir()])
        
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        
        if self.mode not in ('train', 'val', 'test'):
            raise ValueError(f"mode must be 'train', 'val', or 'test', got {self.mode!r}")
            
        self.samples = []
        if not self.class_names:
            raise ValueError(f"No class directories found in {root_dir}.")
        
        for class_name in self.class_names:
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                raise ValueError(f"Class directory {class_dir} does not exist.")
            for ex in self.image_extensions:
                for image_path in class_dir.glob(f"*{ex}"):
                    self.samples.append({
                        'path':image_path,    # like this -> dataset/train/healthy/image1.jpg'
                        'label': self.class_to_idx[class_name]
                    })
        if not self.samples:
            raise ValueError("No images found in the dataset.")
                 
        if transform is not None:
            self.transform = transform
            
        elif self.mode == 'train':
            self.transform = self.default_train_transform()
        else:
            self.transform = self.default_val_transform()
            
    def default_train_transform(self):
        """Define default transformations for training.  Data Augmention"""
        return transforms.Compose([
            transforms.RandomRotation(10),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomResizedCrop((100, 100), scale=(0.8, 1.0)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    def default_val_transform(self):
        """Define default transformations for validation."""
        return transforms.Compose([
            transforms.Resize((100, 100)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    def __len__(self):
        """Return the number of samples in the dataset."""
        return len(self.samples)
    
    
    def __getitem__(self,idx:int):
        """get item function """
        sample = self.samples[idx]
        try:
            image = Image.open(sample['path']).convert('RGB') # to make any img for 3 channels
            if self.transform:
                image = self.transform(image)
            label = sample['label']
            return image,label
        except Exception as e:
            warnings.warn(f"Skipping corrupted image {sample['path']}: {e}")
            fallback_size = getattr(self, "_fallback_shape", (3, 100, 100))  # Default fallback shape
            return torch.zeros(fallback_size), sample['label']
           

In [10]:

train_dataset = FruitsDataset(root_dir=TRAIN_PATH, mode='train')
test_dataset = FruitsDataset(root_dir=TEST_PATH, mode='test')



In [11]:
class CNNModel(nn.Module):
    """
    Simple CNN for image classification.
    Input: (B, 3, 224, 224)
    """

    def __init__(self, num_classes: int):
        super().__init__()

        # Feature Extractor
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),

            # Input:
            # 224x224
            # ↓ Pool
            # 112x112
            # ↓ Pool
            # 56x56

            nn.Linear(64 , 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.ReLU(inplace=True),

            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
model = CNNModel(num_classes=len(train_dataset.class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)



In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print(next(iter(train_loader)))  # Print the first batch of the train_loader

C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train the model for one epoch.

    Returns:
        epoch_loss (float): Average loss per sample.
        epoch_accuracy (float): Classification accuracy.
    """

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        # Move data to device
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Clear gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        batch_size = images.size(0)

        running_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def validate(model, dataloader, criterion, device):
    """
    Evaluate the model.
    """

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predictions = torch.max(outputs, dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
import time

NUM_EPOCHS = 30

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # Train
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )

    # Validation
    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device,
    )

    end = time.time()

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2%} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2%} | "
        f"Time: {(end - start):.2f}s"
    )

NameError: name 'evaluate' is not defined

In [ ]:
# validate(model, train_loader, criterion, device)

In [ ]:
print('Train Dataset Size:', len(train_dataset))
print('Test Dataset Size:', len(test_dataset))

Train Dataset Size: 137221
Test Dataset Size: 45724


In [ ]:
import time

start = time.time()

for i, (images, labels) in enumerate(train_loader):
    if i == 100:
        break

print(f"100 batches loading time: {time.time() - start:.2f} seconds")

100 batches loading time: 15.45 seconds


In [ ]:
"""
Fruit image classification pipeline built on PyTorch.

Includes:
    - FruitsDataset: an ImageFolder-style dataset with corrupted-image filtering
    - build_datasets: resolves an explicit train/val split OR auto-splits a
      single flat "one folder per class" directory
    - CNNModel: a small from-scratch CNN
    - ResNetClassifier: a torchvision ResNet18 backbone (optionally pretrained
      and/or frozen) for transfer learning
    - train_one_epoch / evaluate: training and validation loops with optional
      AMP (mixed precision) and gradient clipping
    - main(): a CLI entry point wiring everything together with checkpointing,
      early stopping, LR scheduling, and reproducibility controls

Example
-------
    # Explicit split: data_dir/train/<class>/*.jpg, data_dir/val/<class>/*.jpg
    python fruits_classifier.py --data-dir ./dataset --epochs 20

    # Flat directory (data_dir/<class>/*.jpg) -> auto 80/20 split
    python fruits_classifier.py --data-dir ./dataset_flat --val-split 0.2

    # Transfer learning
    python fruits_classifier.py --data-dir ./dataset --model resnet18 \
        --pretrained --image-size 224 --normalize imagenet
"""

from __future__ import annotations

import argparse
import logging
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image, UnidentifiedImageError

logger = logging.getLogger(__name__)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
DEFAULT_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


# --------------------------------------------------------------------------- #
# Reproducibility
# --------------------------------------------------------------------------- #
def set_seed(seed: int = 42) -> None:
    """Seed all relevant RNGs for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _worker_init_fn(worker_id: int) -> None:
    """Give each DataLoader worker a distinct, reproducible seed."""
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #
class FruitsDataset(Dataset):
    """
    An ImageFolder-style dataset: expects `root_dir/<class_name>/<image files>`.

    Corrupted or unreadable images are fully decoded and discarded during
    `__init__` (not silently replaced with a fake image at train time), so
    every indexed sample is a verified, correctly labeled image. As a second
    line of defense, `__getitem__` also guards against files that go bad
    on disk between indexing and training (rare, but possible on networked
    filesystems) by logging and skipping to the next sample.
    """

    def __init__(
        self,
        root_dir: str | Path,
        transform: Optional[Callable] = None,
        mode: str = "train",
        image_extensions: Optional[tuple[str, ...]] = None,
        image_size: tuple[int, int] = (100, 100),
        normalize_mean: list[float] = IMAGENET_MEAN,
        normalize_std: list[float] = IMAGENET_STD,
    ) -> None:
        super().__init__()

        if mode not in ("train", "val", "test"):
            raise ValueError(f"mode must be 'train', 'val', or 'test', got {mode!r}")
        self.mode = mode
        
        self.image_size = image_size
        
        self.normalize_mean = normalize_mean
        self.normalize_std = normalize_std

        self.image_extensions = tuple(
            ext.lower() for ext in (image_extensions or DEFAULT_EXTENSIONS)
        )

        self.root_dir = Path(root_dir)
        if not self.root_dir.is_dir():
            raise ValueError(f"Root directory {self.root_dir} does not exist.")

        self.class_names = sorted(
            f.name for f in self.root_dir.iterdir() if f.is_dir()
        )
        if not self.class_names:
            raise ValueError(f"No class directories found in {self.root_dir}.")

        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        self.samples: list[dict] = self._build_and_verify_samples()
        if not self.samples:
            raise ValueError("No valid images found in the dataset.")

        self.transform = transform or (
            self.default_train_transform(image_size, normalize_mean, normalize_std)
            if self.mode == "train"
            else self.default_val_transform(image_size, normalize_mean, normalize_std)
        )

    # -- construction helpers ------------------------------------------------ #
    def _iter_image_paths(self, class_dir: Path):
        """Yield image paths under class_dir, matching extensions case-insensitively."""
        for path in class_dir.iterdir():
            if path.is_file() and path.suffix.lower() in self.image_extensions:
                yield path

    def _build_and_verify_samples(self) -> list[dict]:
        """
        Scan every class directory, fully decode each image to verify it is
        actually readable, and drop any file that is missing/corrupted
        (logging a warning) instead of deferring the failure to training time.
        """
        samples = []
        skipped = 0

        for class_name in self.class_names:
            class_dir = self.root_dir / class_name
            for image_path in self._iter_image_paths(class_dir):
                if self._is_readable(image_path):
                    samples.append(
                        {"path": image_path, "label": self.class_to_idx[class_name]}
                    )
                else:
                    skipped += 1
                    logger.warning("Skipping unreadable image: %s", image_path)

        if skipped:
            logger.info("Skipped %d corrupted/unreadable image(s) during indexing.", skipped)
        return samples

    @staticmethod
    def _is_readable(path: Path) -> bool:
        """Fully decode the image (not just verify()) to catch truncated files."""
        try:
            with Image.open(path) as img:
                img.load()
            return True
        except (UnidentifiedImageError, OSError, ValueError):
            return False

    # -- default transforms --------------------------------------------------- #
    @staticmethod
    def default_train_transform(
        image_size: tuple[int, int] = (100, 100),
        mean: list[float] = IMAGENET_MEAN,
        std: list[float] = IMAGENET_STD,
    ) -> transforms.Compose:
        """Training-time transforms: light augmentation + normalization."""
        return transforms.Compose(
            [
                transforms.RandomRotation(10),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(
                    brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1
                ),
                transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        )

    @staticmethod
    def default_val_transform(
        image_size: tuple[int, int] = (100, 100),
        mean: list[float] = IMAGENET_MEAN,
        std: list[float] = IMAGENET_STD,
    ) -> transforms.Compose:
        """Validation/test-time transforms: deterministic resize + normalization."""
        return transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        )

    # -- Dataset protocol ------------------------------------------------------ #
    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        sample = self.samples[idx]
        try:
            image = Image.open(sample["path"]).convert("RGB")
        except (UnidentifiedImageError, OSError) as exc:
            # Defense-in-depth: the file passed verification at indexing time
            # but has since become unreadable (e.g. removed/corrupted on disk).
            logger.error(
                "Runtime read failure for %s (%s); skipping to next sample.",
                sample["path"], exc,
            )
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            image = self.transform(image)
        return image, sample["label"]


class _TransformSubset(Dataset):
    """Wraps a subset of a FruitsDataset's samples with its own transform.

    Used to give a train split and a val split (drawn from the same flat
    directory) different transforms (augmented vs. deterministic) without
    duplicating the file-scanning/verification work.
    """

    def __init__(self, base: FruitsDataset, indices: list[int], transform: Callable) -> None:
        self.base = base
        self.indices = indices
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
        sample = self.base.samples[self.indices[i]]
        try:
            image = Image.open(sample["path"]).convert("RGB")
        except (UnidentifiedImageError, OSError) as exc:
            logger.error(
                "Runtime read failure for %s (%s); skipping to next sample.",
                sample["path"], exc,
            )
            return self.__getitem__((i + 1) % len(self))
        return self.transform(image), sample["label"]


def build_datasets(
    data_dir: Path,
    val_split: float,
    seed: int,
    image_size: tuple[int, int],
    mean: list[float],
    std: list[float],
) -> tuple[Dataset, Dataset, list[str]]:
    """
    Resolve a train/val dataset pair from `data_dir`.

    If `data_dir/train` and `data_dir/val` both exist, they are used directly
    (each scanned/verified independently, with a sanity check that both sides
    agree on the set of classes). Otherwise `data_dir` is treated as a flat
    `<class>/<images>` directory and randomly split into train/val subsets
    that share the same underlying file index but use different transforms.
    """
    train_dir = data_dir / "train"
    val_dir = data_dir / "val"

    if train_dir.is_dir() and val_dir.is_dir():
        train_ds = FruitsDataset(
            train_dir, mode="train", image_size=image_size,
            normalize_mean=mean, normalize_std=std,
        )
        val_ds = FruitsDataset(
            val_dir, mode="val", image_size=image_size,
            normalize_mean=mean, normalize_std=std,
        )
        if train_ds.class_names != val_ds.class_names:
            raise ValueError(
                f"train/val class mismatch: train={train_ds.class_names}, "
                f"val={val_ds.class_names}"
            )
        return train_ds, val_ds, train_ds.class_names

    if not (0.0 < val_split < 1.0):
        raise ValueError(f"--val-split must be in (0, 1), got {val_split}")

    logger.info(
        "No explicit train/val subfolders found under %s; auto-splitting "
        "flat class directories with val_split=%.2f", data_dir, val_split,
    )
    base = FruitsDataset(
        data_dir, mode="val", image_size=image_size,  # transform overridden per-subset below
        normalize_mean=mean, normalize_std=std,
    )

    rng = random.Random(seed)
    indices = list(range(len(base.samples)))
    rng.shuffle(indices)
    n_val = max(1, int(len(indices) * val_split))
    val_indices, train_indices = indices[:n_val], indices[n_val:]

    train_transform = FruitsDataset.default_train_transform(image_size, mean, std)
    val_transform = FruitsDataset.default_val_transform(image_size, mean, std)

    train_ds = _TransformSubset(base, train_indices, train_transform)
    val_ds = _TransformSubset(base, val_indices, val_transform)
    return train_ds, val_ds, base.class_names


def compute_dataset_stats(
    dataset: FruitsDataset, image_size: tuple[int, int], sample_size: int = 300
) -> tuple[list[float], list[float]]:
    """
    Estimate per-channel mean/std over a random sample of the dataset's raw
    (resized, un-normalized) pixels. Use this instead of ImageNet statistics
    when training from scratch on visually distinct data.
    """
    resize_only = transforms.Compose([transforms.Resize(image_size), transforms.ToTensor()])
    n = min(sample_size, len(dataset.samples))
    sample_indices = random.sample(range(len(dataset.samples)), n)

    pixel_sum = torch.zeros(3)
    pixel_sq_sum = torch.zeros(3)
    pixel_count = 0

    for i in sample_indices:
        img = Image.open(dataset.samples[i]["path"]).convert("RGB")
        tensor = resize_only(img)
        pixel_sum += tensor.sum(dim=(1, 2))
        pixel_sq_sum += (tensor ** 2).sum(dim=(1, 2))
        pixel_count += tensor.shape[1] * tensor.shape[2]

    mean = pixel_sum / pixel_count
    variance = (pixel_sq_sum / pixel_count) - mean ** 2
    std = variance.clamp(min=1e-8).sqrt()
    return mean.tolist(), std.tolist()


# --------------------------------------------------------------------------- #
# Models
# --------------------------------------------------------------------------- #
class CNNModel(nn.Module):
    """
    A compact from-scratch CNN classifier.

    Uses AdaptiveAvgPool2d before the classifier head, so it accepts any
    input spatial resolution without architecture changes.
    """

    def __init__(self, num_classes: int) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


class ResNetClassifier(nn.Module):
    """
    Transfer-learning classifier built on torchvision's ResNet18.

    Recommended when the dataset is small or classes are visually subtle
    (e.g. ripe vs. spoiled fruit): pretrained ImageNet features generalize
    far better than a from-scratch shallow CNN in that regime.
    """

    def __init__(
        self, num_classes: int, pretrained: bool = True, freeze_backbone: bool = False
    ) -> None:
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)  # always trainable

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


def build_model(
    architecture: str, num_classes: int, pretrained: bool = True, freeze_backbone: bool = False
) -> nn.Module:
    """Factory for selecting a model architecture by name."""
    if architecture == "simple":
        return CNNModel(num_classes=num_classes)
    if architecture == "resnet18":
        return ResNetClassifier(
            num_classes=num_classes, pretrained=pretrained, freeze_backbone=freeze_backbone
        )
    raise ValueError(f"Unknown architecture: {architecture!r}")


# --------------------------------------------------------------------------- #
# Train / eval loops
# --------------------------------------------------------------------------- #
@dataclass
class EpochMetrics:
    loss: float
    accuracy: float


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    grad_clip: Optional[float] = None,
    use_amp: bool = False,
    scaler: Optional[torch.amp.GradScaler] = None,
) -> EpochMetrics:
    """Run one training epoch and return average loss and accuracy."""
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return EpochMetrics(loss=running_loss / total, accuracy=correct / total)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> EpochMetrics:
    """Run inference over a dataloader (no gradient updates) and return metrics."""
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return EpochMetrics(loss=running_loss / total, accuracy=correct / total)



def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
    set_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = args.amp and device.type == "cuda"
    if args.amp and not use_amp:
        logger.warning("--amp requested but CUDA is not available; running in full precision.")
    logger.info("Using device: %s", device)

    image_size = (args.image_size, args.image_size)

    # First pass with ImageNet stats to enable indexing; recomputed below if requested.
    mean, std = IMAGENET_MEAN, IMAGENET_STD
    if args.normalize == "computed":
        probe_dir = args.data_dir / "train" if (args.data_dir / "train").is_dir() else args.data_dir
        probe_dataset = FruitsDataset(probe_dir, mode="val", image_size=image_size)
        mean, std = compute_dataset_stats(probe_dataset, image_size)
        logger.info("Computed dataset normalization: mean=%s std=%s", mean, std)

    train_dataset, val_dataset, class_names = build_datasets(
        args.data_dir, args.val_split, args.seed, image_size, mean, std
    )
    num_classes = len(class_names)
    logger.info(
        "Found %d classes: %s | train=%d val=%d",
        num_classes, class_names, len(train_dataset), len(val_dataset),
    )

    train_loader = DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=True,
        num_workers=args.num_workers, pin_memory=(device.type == "cuda"),
        worker_init_fn=_worker_init_fn if args.num_workers > 0 else None,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=args.batch_size, shuffle=False,
        num_workers=args.num_workers, pin_memory=(device.type == "cuda"),
    )

    model = build_model(
        args.model, num_classes, pretrained=args.pretrained, freeze_backbone=args.freeze_backbone
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.lr, weight_decay=args.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, args.epochs + 1):
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            grad_clip=args.grad_clip, use_amp=use_amp, scaler=scaler,
        )
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_metrics.loss)

        logger.info(
            "Epoch %d/%d | train_loss=%.4f train_acc=%.4f | val_loss=%.4f val_acc=%.4f",
            epoch, args.epochs,
            train_metrics.loss, train_metrics.accuracy,
            val_metrics.loss, val_metrics.accuracy,
        )

        if val_metrics.loss < best_val_loss:
            best_val_loss = val_metrics.loss
            epochs_without_improvement = 0
            checkpoint_path = args.output_dir / "best_model.pt"
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_to_idx": {name: i for i, name in enumerate(class_names)},
                    "architecture": args.model,
                    "image_size": image_size,
                    "normalize_mean": mean,
                    "normalize_std": std,
                    "epoch": epoch,
                    "val_loss": val_metrics.loss,
                    "val_accuracy": val_metrics.accuracy,
                },
                checkpoint_path,
            )
            logger.info("Saved new best checkpoint to %s", checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= args.patience:
                logger.info("Early stopping: no val_loss improvement for %d epochs.", args.patience)
                break


if __name__ == "__main__":
    main()

: 